In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
from tqdm import tqdm
import os

def search_transfermarkt_player_by_name(player_name):
    """변경된 HTML 구조에 맞춰 선수 ID를 검색하고 반환합니다."""
    
    search_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'}
    params = {'query': player_name}
    
    try:
        response = requests.get(search_url, headers=headers, params=params)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        first_result = soup.select_one("td.hauptlink a")
        
        if first_result:
            href = first_result.get('href', '')
            player_id_match = re.search(r'/spieler/(\d+)', href)
            if player_id_match:
                return player_id_match.group(1)
        
        return None
        
    except requests.exceptions.RequestException as e:
        # 네트워크 오류가 발생하면 즉시 메시지를 출력합니다.
        tqdm.write(f"-> '{player_name}' 선수 검색 중 네트워크 오류: {e}")
        return None

# --- 1단계: unique_players_list.csv 파일 읽기 ---
input_filename = 'unique_players_list_premier.csv'

if not os.path.exists(input_filename):
    print(f"❌ 오류: '{input_filename}' 파일이 없습니다. 이 파일을 먼저 생성해주세요.")
else:
    print(f"✅ 1단계: '{input_filename}' 파일에서 선수 명단을 읽어옵니다.")
    unique_players_df = pd.read_csv(input_filename)
    print(f"총 {len(unique_players_df)}명의 선수에 대한 ID 검색을 시작합니다.")

    # --- 2단계: 각 선수의 Transfermarkt ID 검색 및 저장 ---
    print("\n✅ 2단계: 각 선수의 Transfermarkt ID를 검색하며 실시간으로 결과를 출력합니다.")
    player_id_data = []
    failed_players = []  # 실패한 선수 이름을 저장할 리스트 생성

    for _, row in tqdm(unique_players_df.iterrows(), total=unique_players_df.shape[0], desc="ID 검색 중"):
        player_name = row['선수명']
        player_id = search_transfermarkt_player_by_name(player_name)
        
        if player_id:
            player_id_data.append({'선수명': player_name, 'Transfermarkt_ID': player_id})
            tqdm.write(f"✅ [성공] {player_name}: ID {player_id} 찾음")
        else:
            tqdm.write(f"❌ [실패] '{player_name}' 선수의 ID를 찾지 못했습니다.")
            failed_players.append(player_name)  # 실패 시, 리스트에 선수 이름 추가
        
        time.sleep(0.5)

    # --- 3단계: 최종 결과물 생성 (전체 작업 요약) ---
    if player_id_data:
        id_df = pd.DataFrame(player_id_data)
        
        output_filename = 'transfermarkt_player_ids_premier.csv'
        id_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
        
        print("\n\n--- [ 최종 요약 ] ---")
        print(f"🎉 모든 작업 완료! 총 {len(id_df)}명의 선수 ID를 찾아 '{output_filename}' 파일에 저장되었습니다.")
        print("\n[ 저장된 결과 (상위 10개) ]")
        print(id_df.head(10).to_string())
    else:
        print("\n\n--- [ 최종 요약 ] ---")
        print("처리된 데이터가 없습니다. 선수 검색 결과를 확인해보세요.")

    # --- 추가된 부분: ID 검색 실패 목록 출력 ---
    if failed_players:
        print("\n\n--- [ ID 검색 실패 목록 ] ---")
        for name in failed_players:
            print(f"- {name}")
    else:
        print("\n\n✨ 모든 선수의 ID를 성공적으로 찾았습니다!")


✅ 1단계: 'unique_players_list_premier.csv' 파일에서 선수 명단을 읽어옵니다.
총 879명의 선수에 대한 ID 검색을 시작합니다.

✅ 2단계: 각 선수의 Transfermarkt ID를 검색하며 실시간으로 결과를 출력합니다.


ID 검색 중:   0%|          | 0/879 [00:01<?, ?it/s]   

✅ [성공] aaron connolly: ID 434207 찾음


ID 검색 중:   0%|          | 1/879 [00:02<24:53,  1.70s/it]   

✅ [성공] aaron cresswell: ID 92571 찾음


...

✅ [성공] édinson cavani: ID 48280 찾음


ID 검색 중: 100%|█████████▉| 878/879 [42:01<00:02,  2.32s/it]   

✅ [성공] étienne capoue: ID 63494 찾음


ID 검색 중: 100%|██████████| 879/879 [42:02<00:00,  2.87s/it]



--- [ 최종 요약 ] ---
🎉 모든 작업 완료! 총 862명의 선수 ID를 찾아 'transfermarkt_player_ids_premier.csv' 파일에 저장되었습니다.

[ 저장된 결과 (상위 10개) ]
                  선수명 Transfermarkt_ID
0      aaron connolly           434207
1     aaron cresswell            92571
2        aaron hickey           591949
3        aaron lennon            14221
4          aaron mooy           123951
5      aaron ramsdale           427568
6        aaron ramsey            50057
7   aaron wan-bissaka           477758
8  abdoulaye doucouré           127187
9      adam armstrong           250426


--- [ ID 검색 실패 목록 ] ---
- ederson moraes
- fabian schär
- fabinho
- fabio
- fabio borini
- fabio silva
- facundo buonanotte
- federico fernández
- felipe anderson
- fernando llorente
- ferran torres
- ibrahim didier ndong
- jens-lys cajuste
- kristoffer vassbakk ajer
- oliver mcburnie
- rayan ait nouri
- thomas ince
